# Laboratório — Dropout do zero

Implementaremos *inverted dropout* em NumPy puro, verificaremos seus momentos e gradientes e encerraremos com uma seleção de `keep_prob` que preserva um teste reservado.

- Python >= 3.11
- NumPy >= 1.26
- Matplotlib >= 3.8
- seed principal: `20260921`

Não usamos autograd nem implementações prontas de redes neurais.

## 1. Ambiente e reprodutibilidade

Uma seed fixa inicializa o stream; ela não deve ser reiniciada dentro de cada forward.

In [ ]:
import copy
import platform
import warnings

import matplotlib
import matplotlib.pyplot as plt
import numpy as np

SEED = 20260921
rng = np.random.default_rng(SEED)

print("Python:", platform.python_version())
print("NumPy:", np.__version__)
print("Matplotlib:", matplotlib.__version__)
print("Seed:", SEED)

assert tuple(map(int, np.__version__.split(".")[:2])) >= (1, 26)
assert tuple(map(int, matplotlib.__version__.split(".")[:2])) >= (3, 8)

## 2. Forward e backward explícitos

O cache guarda exatamente a máscara usada no forward. No modo de inferência, a função é identidade e não precisa de RNG.

In [ ]:
def dropout_forward(a, keep_prob, rng=None, training=True, mask=None):
    if not 0 < keep_prob <= 1:
        raise ValueError("keep_prob deve estar em (0, 1]")
    if not training or keep_prob == 1:
        return a.copy(), None
    if mask is None:
        if rng is None:
            raise ValueError("modo de treino requer rng ou máscara explícita")
        mask = rng.random(a.shape) < keep_prob
    mask = np.asarray(mask, dtype=bool)
    if mask.shape != a.shape:
        raise ValueError("máscara e ativação devem ter o mesmo shape")
    return a * mask / keep_prob, (mask, keep_prob)


def dropout_backward(upstream, cache):
    if cache is None:
        return upstream.copy()
    mask, keep_prob = cache
    if mask.shape != upstream.shape:
        raise ValueError("máscara e gradiente devem ter o mesmo shape")
    return upstream * mask / keep_prob

## 3. Contratos elementares

Testamos o exemplo da aula, a identidade em \(q=1\), o backward e entradas inválidas.

In [ ]:
a = np.array([[1.0, -2.0, 3.0]])
fixed_mask = np.array([[False, True, True]])
out, cache = dropout_forward(a, 0.5, training=True, mask=fixed_mask)
upstream = np.array([[1.0, 4.0, -2.0]])
downstream = dropout_backward(upstream, cache)

identity_out, identity_cache = dropout_forward(a, 1.0, rng=rng, training=True)
identity_grad = dropout_backward(upstream, identity_cache)

caught = []
for bad_q in (0.0, -0.2, 1.1):
    try:
        dropout_forward(a, bad_q, rng=rng)
    except ValueError:
        caught.append(bad_q)

try:
    dropout_forward(a, 0.5, training=True, mask=np.ones((1, 1), dtype=bool))
except ValueError:
    shape_error_caught = True
else:
    shape_error_caught = False

print("Forward q=0,5:", out)
print("Backward:", downstream)
print("Valores inválidos detectados:", caught)

assert np.array_equal(out, [[0.0, -4.0, 6.0]])
assert np.array_equal(downstream, [[0.0, 8.0, -4.0]])
assert np.array_equal(identity_out, a)
assert np.array_equal(identity_grad, upstream)
assert identity_cache is None
assert len(caught) == 3 and shape_error_caught

## 4. Esperança e variância

Para \(M\sim\mathrm{Bernoulli}(q)\), inverted dropout prevê

\[
\mathbb E[MA/q]=A,\qquad
\operatorname{Var}(MA/q)=A^2(1-q)/q.
\]

Estimaremos ambos com 200 mil máscaras independentes.

In [ ]:
values = np.array([1.0, -2.0, 0.5])
keep_prob = 0.65
moment_rng = np.random.default_rng(SEED + 1)
masks = moment_rng.random((200_000, values.size)) < keep_prob
samples = masks * values / keep_prob

empirical_mean = samples.mean(axis=0)
empirical_var = samples.var(axis=0)
theoretical_var = values**2 * (1 - keep_prob) / keep_prob
mean_error = float(np.max(np.abs(empirical_mean - values)))
relative_var_error = float(np.max(np.abs(empirical_var - theoretical_var) / theoretical_var))

print("Média empírica:", empirical_mean)
print("Média teórica:", values)
print("Variância empírica:", empirical_var)
print("Variância teórica:", theoretical_var)
print(f"Erro máximo da média: {mean_error:.6f}")
print(f"Erro relativo máximo da variância: {relative_var_error:.6f}")

assert mean_error < 0.01
assert relative_var_error < 0.015

**Descrição do gráfico:** histogramas para uma ativação unitária mostram massa em zero e em \(1/q\). Ao reduzir \(q\), a massa sobrevivente se afasta de 1 e a variância aumenta.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(11, 3.4), sharey=True)
for ax, q in zip(axes, (0.9, 0.7, 0.5)):
    local_rng = np.random.default_rng(SEED + int(q * 100))
    draws = (local_rng.random(20_000) < q) / q
    ax.hist(draws, bins=20, color="#5B5FEF", alpha=0.85)
    ax.axvline(1.0, color="black", linestyle="--", label="esperança")
    ax.set(title=f"q={q}", xlabel="ativação após dropout")
axes[0].set_ylabel("frequência")
axes[0].legend()
plt.tight_layout()
plt.show()

## 5. Máscara por exemplo e unidade

Nesta MLP densa, a máscara deve ter shape \((B,H)\). Uma máscara \((1,H)\) compartilharia decisões pelo lote via broadcasting.

In [ ]:
batch = np.ones((64, 128))
mask_rng = np.random.default_rng(SEED + 2)
masked_batch, (batch_mask, _) = dropout_forward(batch, 0.7, rng=mask_rng, training=True)
unique_rows = np.unique(batch_mask, axis=0).shape[0]
retention = float(batch_mask.mean())

shared_mask = (np.random.default_rng(SEED + 2).random((1, 128)) < 0.7)
shared_result = batch * shared_mask / 0.7

print("Shape da máscara correta:", batch_mask.shape)
print("Linhas distintas:", unique_rows)
print(f"Retenção observada: {retention:.6f}")

assert batch_mask.shape == batch.shape == masked_batch.shape
assert unique_rows > 60
assert abs(retention - 0.7) < 0.03
assert np.unique(shared_result, axis=0).shape[0] == 1

## 6. Estado do RNG e retomada

Copiar o estado do bit generator deve reproduzir a sequência futura. Reiniciar uma seed em toda chamada, por outro lado, repete a mesma máscara indefinidamente.

In [ ]:
stream = np.random.default_rng(SEED + 3)
_ = stream.random((5, 7))
saved_state = copy.deepcopy(stream.bit_generator.state)
future_a = stream.random((4, 9)) < 0.6

restored = np.random.default_rng()
restored.bit_generator.state = copy.deepcopy(saved_state)
future_b = restored.random((4, 9)) < 0.6

repeated_a = np.random.default_rng(99).random((4, 9)) < 0.6
repeated_b = np.random.default_rng(99).random((4, 9)) < 0.6

print("Retomada reproduz próxima máscara:", np.array_equal(future_a, future_b))
print("Reinicialização repete máscara:", np.array_equal(repeated_a, repeated_b))

assert np.array_equal(future_a, future_b)
assert np.array_equal(repeated_a, repeated_b)

## 7. Inferência determinística não consome aleatoriedade

Duas avaliações devem ser idênticas e preservar o estado do RNG. Duas chamadas de treino devem, em geral, usar máscaras diferentes.

In [ ]:
mode_rng = np.random.default_rng(SEED + 4)
probe = np.arange(24, dtype=float).reshape(4, 6)
state_before = copy.deepcopy(mode_rng.bit_generator.state)
eval_a, _ = dropout_forward(probe, 0.5, rng=mode_rng, training=False)
eval_b, _ = dropout_forward(probe, 0.5, rng=mode_rng, training=False)
state_after = copy.deepcopy(mode_rng.bit_generator.state)

train_a, _ = dropout_forward(probe, 0.5, rng=mode_rng, training=True)
train_b, _ = dropout_forward(probe, 0.5, rng=mode_rng, training=True)

print("Inferências idênticas:", np.array_equal(eval_a, eval_b))
print("Treinos idênticos:", np.array_equal(train_a, train_b))

assert np.array_equal(eval_a, probe)
assert np.array_equal(eval_a, eval_b)
assert state_before == state_after
assert not np.array_equal(train_a, train_b)

## 8. MLP binária com dropout oculto

O contrato é afim → tanh → dropout → afim → BCE com logits. O backward reutiliza a máscara do cache.

In [ ]:
def sigmoid_stable(z):
    out = np.empty_like(z, dtype=float)
    positive = z >= 0
    out[positive] = 1.0 / (1.0 + np.exp(-z[positive]))
    ez = np.exp(z[~positive])
    out[~positive] = ez / (1.0 + ez)
    return out


def bce_with_logits(logits, targets):
    return float(np.mean(np.maximum(logits, 0) - logits * targets + np.log1p(np.exp(-np.abs(logits)))))


def init_params(input_dim, hidden_dim, seed):
    local_rng = np.random.default_rng(seed)
    return {
        "W1": local_rng.normal(0, np.sqrt(1 / input_dim), (input_dim, hidden_dim)),
        "b1": np.zeros(hidden_dim),
        "W2": local_rng.normal(0, np.sqrt(1 / hidden_dim), (hidden_dim, 1)),
        "b2": np.zeros(1),
    }


def mlp_forward(params, x, keep_prob=1.0, rng=None, training=False, mask=None):
    z1 = x @ params["W1"] + params["b1"]
    h = np.tanh(z1)
    h_drop, drop_cache = dropout_forward(h, keep_prob, rng=rng, training=training, mask=mask)
    logits = h_drop @ params["W2"] + params["b2"]
    return logits, (x, h, h_drop, drop_cache)


def mlp_backward(params, cache, logits, targets):
    x, h, h_drop, drop_cache = cache
    dlogits = (sigmoid_stable(logits) - targets) / len(targets)
    grads = {}
    grads["W2"] = h_drop.T @ dlogits
    grads["b2"] = dlogits.sum(axis=0)
    dh_drop = dlogits @ params["W2"].T
    dh = dropout_backward(dh_drop, drop_cache)
    dz1 = dh * (1 - h**2)
    grads["W1"] = x.T @ dz1
    grads["b1"] = dz1.sum(axis=0)
    return grads

## 9. Gradient checking: máscara congelada versus reamostrada

Usaremos uma pequena rede. As diferenças centrais corretas reutilizam a mesma máscara; a contraprova amostra sub-redes distintas nos lados \(+\) e \(-\).

In [ ]:
check_rng = np.random.default_rng(SEED + 5)
x_check = check_rng.normal(size=(6, 2))
y_check = (check_rng.random((6, 1)) > 0.5).astype(float)
p_check = init_params(2, 4, SEED + 6)
frozen_mask = check_rng.random((6, 4)) < 0.7

logits, cache = mlp_forward(p_check, x_check, 0.7, training=True, mask=frozen_mask)
analytic = mlp_backward(p_check, cache, logits, y_check)


def loss_with_mask(params, mask):
    z, _ = mlp_forward(params, x_check, 0.7, training=True, mask=mask)
    return bce_with_logits(z, y_check)


h = 1e-5
coordinates = [("W1", (0, 2)), ("b1", (3,)), ("W2", (1, 0)), ("b2", (0,))]
fixed_errors = []
for name, index in coordinates:
    plus = {k: v.copy() for k, v in p_check.items()}
    minus = {k: v.copy() for k, v in p_check.items()}
    plus[name][index] += h
    minus[name][index] -= h
    numerical = (loss_with_mask(plus, frozen_mask) - loss_with_mask(minus, frozen_mask)) / (2 * h)
    error = abs(numerical - analytic[name][index])
    fixed_errors.append(error)
    print(f"{name}{index}: analítico={analytic[name][index]:.9f} numérico={numerical:.9f} erro={error:.3e}")

bad_rng = np.random.default_rng(SEED + 7)
name, index = "W1", (0, 2)
plus = {k: v.copy() for k, v in p_check.items()}
minus = {k: v.copy() for k, v in p_check.items()}
plus[name][index] += h
minus[name][index] -= h
mask_plus = bad_rng.random((6, 4)) < 0.7
mask_minus = bad_rng.random((6, 4)) < 0.7
bad_numerical = (loss_with_mask(plus, mask_plus) - loss_with_mask(minus, mask_minus)) / (2 * h)
bad_error = abs(bad_numerical - analytic[name][index])

max_fixed_error = float(max(fixed_errors))
print(f"Erro máximo com máscara congelada: {max_fixed_error:.3e}")
print(f"Erro com máscaras reamostradas: {bad_error:.3e}")

assert max_fixed_error < 1e-9
assert bad_error > 100

## 10. Dados sintéticos e protocolo

Geramos treino pequeno com 18% de rótulos invertidos. Validação e teste são populações sintéticas independentes, sem inversão. O teste é armazenado, mas só uma função selada pode consultá-lo.

In [ ]:
def make_dataset(n, seed, label_flip=0.0):
    local_rng = np.random.default_rng(seed)
    x = local_rng.uniform(-2, 2, size=(n, 2))
    score = np.sin(2.5 * x[:, 0]) + np.cos(2.8 * x[:, 1]) + 0.35 * x[:, 0] * x[:, 1]
    y = (score > 0).astype(float)[:, None]
    if label_flip:
        indices = local_rng.choice(n, int(n * label_flip), replace=False)
        y[indices] = 1 - y[indices]
    return x, y


x_train, y_train = make_dataset(72, SEED + 10, 0.18)
x_val, y_val = make_dataset(1200, SEED + 11)
sealed_test = make_dataset(1200, SEED + 12)
test_queries = 0

print("Treino:", x_train.shape, "positivos:", int(y_train.sum()))
print("Validação:", x_val.shape, "positivos:", int(y_val.sum()))
print("Teste reservado:", sealed_test[0].shape)

assert x_train.shape == (72, 2)
assert x_val.shape == sealed_test[0].shape == (1200, 2)
assert not np.array_equal(x_train[:72], x_val[:72])

## 11. Laço de treino e avaliação

A avaliação sempre usa `training=False`. Históricos registram BCE de dados, não uma loss estocástica isolada.

In [ ]:
INITIAL_PARAMS = init_params(2, 96, SEED + 20)


def evaluate(params, x, y):
    logits, _ = mlp_forward(params, x, training=False)
    return bce_with_logits(logits, y), float(np.mean((logits >= 0) == y))


def train_model(keep_prob, steps=6000, learning_rate=0.08, dropout_seed=SEED + 30):
    params = {k: v.copy() for k, v in INITIAL_PARAMS.items()}
    drop_rng = np.random.default_rng(dropout_seed)
    history = {"step": [], "train": [], "val": []}
    for step in range(steps + 1):
        if step % 200 == 0:
            history["step"].append(step)
            history["train"].append(evaluate(params, x_train, y_train)[0])
            history["val"].append(evaluate(params, x_val, y_val)[0])
        if step == steps:
            break
        logits, cache = mlp_forward(params, x_train, keep_prob, rng=drop_rng, training=True)
        grads = mlp_backward(params, cache, logits, y_train)
        for name in params:
            params[name] -= learning_rate * grads[name]
    return params, history


smoke_params, smoke_history = train_model(0.8, steps=5)
assert len(smoke_history["step"]) == 1
assert all(np.isfinite(v).all() for v in smoke_params.values())
print("Smoke test do laço: aprovado")

## 12. Seleção de \(q\) somente na validação

Todos os candidatos partem dos mesmos parâmetros e usam o mesmo orçamento. Isso é uma ablação controlada; não prova superioridade universal.

In [ ]:
keep_candidates = (1.0, 0.9, 0.7, 0.5)
models = {}
histories = {}
validation_scores = {}

for q in keep_candidates:
    params, history = train_model(q)
    train_loss, train_acc = evaluate(params, x_train, y_train)
    val_loss, val_acc = evaluate(params, x_val, y_val)
    models[q] = params
    histories[q] = history
    validation_scores[q] = val_loss
    print(
        f"q={q:.1f} | treino BCE={train_loss:.6f}, acc={train_acc:.6f} "
        f"| validação BCE={val_loss:.6f}, acc={val_acc:.6f}"
    )

assert test_queries == 0
assert all(np.isfinite(list(validation_scores.values())))

**Descrição do gráfico:** curvas determinísticas de treino e validação para cada probabilidade de retenção. \(q=1\) permite ajuste mais agressivo do pequeno treino ruidoso; valores menores adicionam ruído e elevam a loss de treino.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2), sharex=True)
for q in keep_candidates:
    axes[0].plot(histories[q]["step"], histories[q]["train"], label=f"q={q}")
    axes[1].plot(histories[q]["step"], histories[q]["val"], label=f"q={q}")
axes[0].set(title="Treino", xlabel="step", ylabel="BCE determinística")
axes[1].set(title="Validação", xlabel="step", ylabel="BCE determinística")
for ax in axes:
    ax.grid(alpha=0.25)
    ax.legend()
plt.tight_layout()
plt.show()

## 13. Escolha final e uma única consulta ao teste

O menor BCE de validação determina \(q\). Somente depois dessa decisão abrimos o teste.

In [ ]:
selected_q = min(validation_scores, key=validation_scores.get)
selected_model = models[selected_q]


def evaluate_test_once(params):
    global test_queries
    if test_queries != 0:
        raise RuntimeError("o teste reservado já foi consultado")
    test_queries += 1
    return evaluate(params, *sealed_test)


test_loss, test_accuracy = evaluate_test_once(selected_model)
second_query_blocked = False
try:
    evaluate_test_once(selected_model)
except RuntimeError:
    second_query_blocked = True

print("Scores de validação:", validation_scores)
print("q selecionado:", selected_q)
print(f"Teste reservado: BCE={test_loss:.6f}, acurácia={test_accuracy:.6f}")
print("Segunda consulta bloqueada:", second_query_blocked)

assert selected_q in keep_candidates
assert validation_scores[selected_q] == min(validation_scores.values())
assert validation_scores[selected_q] < validation_scores[1.0]
assert test_loss < 0.7
assert test_queries == 1 and second_query_blocked

## 14. Auditoria final

Os contratos cobrem probabilidade, shapes, identidade, momentos, RNG, modos, gradient checking, protocolo de seleção e lacre do teste.

In [ ]:
checks = {
    "dependencias": tuple(map(int, np.__version__.split(".")[:2])) >= (1, 26),
    "q_invalido": len(caught) == 3,
    "shape_invalido": shape_error_caught,
    "identidade_forward": np.array_equal(identity_out, a),
    "identidade_backward": np.array_equal(identity_grad, upstream),
    "esperanca": mean_error < 0.01,
    "variancia": relative_var_error < 0.015,
    "mascara_independente": unique_rows > 60,
    "retomada_rng": np.array_equal(future_a, future_b),
    "inferencia_deterministica": np.array_equal(eval_a, eval_b),
    "inferencia_sem_rng": state_before == state_after,
    "gradiente_congelado": max_fixed_error < 1e-9,
    "contraprova_estocastica": bad_error > 100,
    "selecao_validacao": validation_scores[selected_q] == min(validation_scores.values()),
    "teste_unico": test_queries == 1,
    "segunda_consulta_bloqueada": second_query_blocked,
}
assert all(checks.values())
print(f"Auditoria final: {sum(checks.values())}/{len(checks)} grupos aprovados")

## Conclusões

1. A média empírica e a variância reproduziram as fórmulas do inverted dropout.
2. Inferência foi determinística e não consumiu o stream aleatório.
3. Retomar o estado do RNG reproduziu a máscara futura.
4. Gradient checking funcionou com máscara congelada e falhou de forma evidente com máscaras reamostradas.
5. O valor de \(q\) foi escolhido pelo BCE de validação; o teste reservado foi consultado uma única vez.

O resultado experimental vale para esta MLP, estes dados sintéticos e este orçamento. Ele demonstra o mecanismo e o protocolo, não que dropout sempre melhora qualquer rede.